<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.1-power-grid-stability-estimation/Ex12.1_03_dynamic_pinn_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.1 · Notebook 03 — Letting the Grid Move

**Paired with L12.1 · Power Grid Stability Estimation**

**Prerequisite: notebooks 00–02.**

Everything so far assumed a steady state. Now a line trips, the machines swing,
and the state becomes a trajectory.

### What changes

The state is δ(t) and ω(t) for each machine, represented by a network taking t
as input. Automatic differentiation supplies the time derivatives — the same
machinery as L8, with time instead of space.

The initial condition is built into the trial solution rather than penalised:

$$\delta(t) = \delta_0 + \left(1 - e^{-t/\tau}\right)\,\mathrm{net}(t)$$

so δ(0) = δ₀ exactly, for any weights. That is the L7.2 habit, and it removes a
failure mode instead of discouraging it.

### The physics

$$\frac{2H_i}{\omega_s}\ddot{\delta}_i + D_i\dot{\delta}_i
= P_{m,i} - P_{e,i}$$

integrated in first-order form, never as written above — a second-order ODE is
not what you hand an optimiser.

### The machines

Machine 0 is the Nordic system seen through the Swedish link — a very large
inertia that barely moves, an infinite bus in all but name. Machine 1 is the
local plant, and it is the one that swings.

---

## 0 · Setup

### What is measured, what is not, and why this is not supervised learning

This is the notebook where the phrase *physics and data loss* becomes
misleading, so it is worth being exact about what each term does.

| quantity | measured? | what constrains it |
|---|---|---|
| frequency ω | yes, at scattered instants | the data term, at those instants only |
| load angle δ | **never, anywhere** | the swing equation residual, everywhere |

**There are no labels for δ.** Not sparse labels, not noisy labels. None. The
network outputs a trajectory for δ and nothing in the objective compares it to
a known answer, because no known answer exists.

Contrast the load flow in notebook 00, where a Newton-Raphson solver produced
the true state and everything downstream could be scored against it. Nothing
plays that role here.

So what makes δ come out right? The swing equation ties δ to ω through its
derivatives. The data term pins ω where it was measured. The residual then
propagates that constraint into δ, at every instant, including the ones between
measurements. **The physics is what turns a measured quantity into an
unmeasured one.**

### Where this sits between the two things you already know

It is worth placing this against the two extremes seen earlier in the course.

- **Purely supervised**, as in exercise sets 3 to 6: every output has a label,
  the loss is a comparison, and the model learns the mapping the labels
  describe. It cannot be asked about a quantity nobody labelled.
- **Purely physics-driven**, as in exercise set 7: no data at all, and the loss
  is a residual. The equation and its boundary conditions determine the answer
  completely.
- **This notebook** is neither. The data is real but partial, and it covers one
  of the two state variables. The physics supplies the rest. Neither term alone
  would determine the trajectory: without the data the swing equation has a
  family of solutions, and without the physics the measured ω says nothing
  about δ.

That combination is the thing worth taking from this exercise, and it is the
situation an engineer is usually in. Some quantities are instrumented and the
ones that matter are not.

### One honest limitation

This works cleanly for two machines. On a real network the number of unknown
angles grows with the buses, the swing equations couple through the full
admittance matrix, and the residual becomes correspondingly harder to drive
down. Nothing here demonstrates that it scales, and the literature on that
question is still active.


In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.1-power-grid-stability-estimation/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
ref = pb.load("00_reference")
V, th, P, Q = ref["V"], ref["th"], ref["P"], ref["Q"]
Y = pb.build_ybus()

machines = pb.Machines()
Yred_pre = pb.reduced_admittance(Y, machines, V, th, P, Q)

Y_post = pb.build_ybus(outage=0)              # the severe contingency
V_p, th_p, ok_p, _ = pb.solve_power_flow(Y_post, P, Q)
Yred_post = pb.reduced_admittance(Y_post, machines, V_p, th_p, P, Q)
print(f"post-outage power flow converged: {ok_p}")

delta0, Pm = pb.equilibrium(machines, Yred_pre)
print(f"equilibrium angles (deg): {np.degrees(delta0).round(3)}")
print(f"mechanical power (p.u.):  {Pm.round(3)}")
print(f"H = {machines.H}  (machine 1 is the one that swings)")

**Expected output**

> Post-outage power flow converges. Equilibrium angles about `[0, 2]` degrees.
>
> **Why the equilibrium matters:** start anywhere else and the machines swing
> *before* the fault, which makes every plot afterwards unreadable. If your angles
> are not small and steady in the pre-fault window, this is why.
>
> Note that `pb.equilibrium` **writes back** to `machines.Pm` — machine 0's
> mechanical power is whatever it turns out to be delivering, about `-0.481`
> rather than the placeholder `-1.0` it was constructed with. Notebook 04 fits
> the same model to the same trajectory, so it needs that number, which is why
> it is saved at the bottom of this notebook.

## 1 · Simulate the truth

Integrate through pre-fault, fault-on and post-fault with RK4. This is the
reference trajectory — the answer the estimator is trying to recover from
frequency measurements alone.

In [ ]:
t_fault, t_clear = 0.5, 0.6
T, D, W = pb.simulate_swing(machines, Yred_pre, Yred_post,
                            t_end=5.0, t_fault=t_fault, t_clear=t_clear,
                            delta0=delta0)
fig = pb.plot_swing(T, D, W, t_fault, t_clear,
                    title="truth: line 0-1 out, cleared in 0.1 s"); plt.show()

sep = np.degrees(D.max(axis=1) - D.min(axis=1))
f = W / (2 * np.pi)
print(f"max angle separation: {sep.max():.1f} deg")
print(f"frequency range: {f.min():.2f} to {f.max():.2f} Hz")

**Expected output**

> A visible swing during the shaded fault window, then a damped oscillation
> settling to a new equilibrium.
>
> Max separation around **28 degrees**; frequency between about **49.3 and
> 50.7 Hz**. This is a comfortably stable case — the separation stays well under
> the 180 degrees that counts as loss of synchronism — and you push it towards
> instability in the TODO below.

## 2 · The critical clearing time

The longest fault the system survives. Computed here by bisection on the
simulation, which is crude but honest, and it is the number a stability
assessment ultimately wants.

In [ ]:
cct = pb.critical_clearing_time(machines, Yred_pre, Yred_post, delta0, hi=0.9)
print(f"critical clearing time: {cct:.3f} s")
print(f"we cleared in {t_clear - t_fault:.3f} s — "
      f"{100*(t_clear-t_fault)/cct:.0f}% of the way to the limit")

**Expected output**

> **CCT around 0.330 s** for this contingency. Clearing in 0.1 s uses about 30%
> of the available margin.
>
> Other outages in this network are far milder — some have no critical clearing
> time at all within the search range, because the system survives any clearing
> time. That spread is realistic and is what makes contingency *screening* a real
> problem.

## 3 · Estimate the trajectory from frequency alone

Now the exercise proper. You are given frequency measurements at the machine —
noisy, sampled at a PMU rate — and must recover the whole trajectory, with the
swing equation as the residual.

The collocation times are **not** the sample times. 600 points spread over the
window carry the residual; 251 noisy samples carry the data term. A trajectory
that fits the samples but violates the physics between them is exactly what the
residual exists to catch, and TODO 2 asks you to go and look.

In [ ]:
rate = 50.0                                   # PMU reporting rate, Hz
idx = np.arange(0, len(T), max(int(round(1.0/(rate*(T[1]-T[0])))), 1))
t_obs = T[idx]
f_obs = (W[idx] / (2*np.pi)) + 0.002*np.random.default_rng(11).standard_normal(W[idx].shape)

net, hist = pb.dynamic_pinn(t_obs, f_obs, machines, Yred_post,
                            delta0, W[0], lam_dyn=1.0,
                            adam_steps=3000, lbfgs_steps=15)
fig = pb.plot_loss(hist, "dynamic estimator"); plt.show()

**Expected output**

> Both terms fall. The `dyn` term is the one to watch: if it plateaus high while
> `data` keeps falling, the trajectory is fitting the measurements while
> violating the physics, and λ needs raising.
>
> *(Not executed by the author — see the note in notebook 02.)*

## TODO 1 — push it to instability

Increase the clearing time towards the CCT and past it. Find where the machine
loses synchronism, and compare it to the `critical_clearing_time` above.

In [ ]:
# TODO 1 --- sweep the clearing time past the limit ------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  pb.simulate_swing(machines, Yred_pre, Yred_post, t_end=5.0, t_fault=t_fault, t_clear=t_fault + tc, delta0=delta0)
#   line 2  ->  float(np.degrees(np.abs(D2[:, 1] - D2[:, 0]).max()))     the largest angle separation, degrees
tcs = np.linspace(0.05, 0.45, 9)
seps = []
for tc in tcs:
    T2, D2, W2 = ...                              # <- pb.simulate_swing(machines, Yred_pre, Yred_post, t_end=5.0, t_fault=t_fault, t_clear=t_fault + tc, delta0=delta0)
    seps.append(...)                              # <- float(np.degrees(np.abs(D2[:, 1] - D2[:, 0]).max()))
    print(f"  clearing after {tc:.3f} s: max separation {seps[-1]:7.1f} degrees")

fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.plot(tcs, seps, "o-", lw=1.8)
ax.axvline(cct, color="#d94f2b", ls="--", label=f"bisection CCT = {cct:.3f} s")
ax.set_xlabel("clearing time after fault [s]"); ax.set_ylabel("max angle separation [deg]")
ax.set_yscale("log"); ax.legend(frameon=False); ax.grid(alpha=0.25, which="both")
plt.show()
# ------------------------------------------------------------------------------

## TODO 2 — the residual as a self-check

Evaluate the swing residual on your fitted trajectory at times **between** the
measurement samples. A trajectory that fits the samples but violates the physics
between them is exactly the failure the residual exists to catch.

In [ ]:
# TODO 2 --- the swing residual between the samples ------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  to_tensor(np.linspace(0.0, float(t_obs.max()), 2000).reshape(-1, 1), requires_grad=True)
#   line 2  ->  grad(d_c[:, 1:2], t_dense) - (w_c[:, 1:2] - machines.ws)      d(delta)/dt = omega - omega_s, machine 1
t_dense = ...                                     # <- to_tensor(np.linspace(0.0, float(t_obs.max()), 2000).reshape(-1, 1), requires_grad=True)
d_c, w_c = net(t_dense)
r1 = ...                                          # <- grad(d_c[:, 1:2], t_dense) - (w_c[:, 1:2] - machines.ws)

tt = to_numpy(t_dense).ravel(); rr = np.abs(to_numpy(r1)).ravel()
fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.semilogy(tt, rr, lw=1.2)
ax.axvspan(t_fault, t_clear, color="#f4a300", alpha=0.2, label="fault on")
ax.set_xlabel("t [s]"); ax.set_ylabel("|kinematic residual|, machine 1"); ax.legend(frameon=False); ax.grid(alpha=0.25, which="both")
plt.show()
print(f"largest residual at t = {tt[np.argmax(rr)]:.3f} s")
# Where is it largest -- the fault, the clearing, or neither? What would a residual that is
# large everywhere tell you?
# ------------------------------------------------------------------------------

---

### Two machines is not a real network

This works here, and it is fair to ask how far it carries. Two machines give
one relative angle; a real system has hundreds, and three things get harder
at once. The state grows, so the network must output many coupled
trajectories from the same scattered frequency record. The observability
question returns in a harder form: a handful of PMUs cannot pin down every
machine's angle, and the residual can only propagate a constraint to angles
the dynamics actually couple to a measurement. And the reduced admittance
matrix stops being a 2-by-2 you can print. The method does not break at
scale, but it stops being this simple, and scaling it is a research topic
rather than an exercise. What survives unchanged is the idea: physics turning
a measured quantity into an unmeasured one.


In [ ]:
pb.save("03_dynamic", T=T, D=D, W=W, t_obs=t_obs, f_obs=f_obs,
        cct=np.array([cct]), delta0=delta0, Pm=Pm, omega0=W[0])
print("\nnotebook 03 complete — go to 04_inertia")